# Neural Deconstruction Search for CVRP: Paper Reproduction

**Paper:** *Neural Deconstruction Search for Vehicle Routing Problems* (Hottung et al., 2025)

**Scope:**

## Setup

In [13]:
import torch
from torchinfo import summary

from core.models.nds.model import NDSModel
from core.models.nds.seed_sampler import sample_seed
from core.problems.cvrp import CVRPConfig, CVRPSampler
from core.problems.cvrp.solve import one_customer_per_route

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"PyTorch version: {torch.__version__}")
print(f"Using device: {DEVICE}")

PyTorch version: 2.10.0+cu130
Using device: cuda


In [14]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Data

In [15]:
example_sampler = CVRPSampler(CVRPConfig.for_n(20))

example_batch = example_sampler(4)
print(example_batch)

CVRPInstance(
    capacity=Tensor(shape=torch.Size([4, 1]), device=cpu, dtype=torch.float32, is_shared=False),
    demand=Tensor(shape=torch.Size([4, 21]), device=cpu, dtype=torch.float32, is_shared=False),
    locations=Tensor(shape=torch.Size([4, 21, 2]), device=cpu, dtype=torch.float32, is_shared=False),
    batch_size=torch.Size([4]),
    device=None,
    is_shared=False)


## Model Architecture

In [16]:
D_H = 128
N_HEADS = 8
D_FF = 512

Z_DIM = 10
D_POLY = 256
M = 15

### Encoder

**Initial Node Embedding**

In [17]:
from core.models.nds.encoder import CVRPInitialNodeEmbedding

depot_location = example_batch.depot_location
customer_features = example_batch.customer_features

print(f"Depot location shape: {depot_location.shape}")
print(f"Customer features shape: {customer_features.shape}")

initial_embedding = CVRPInitialNodeEmbedding(d_h=D_H)
initial_node_embeddings = initial_embedding(depot_location, customer_features)
print(f"Initial node embeddings shape: {initial_node_embeddings.shape}")

Depot location shape: torch.Size([4, 1, 2])
Customer features shape: torch.Size([4, 20, 3])
Initial node embeddings shape: torch.Size([4, 21, 128])


**Attention Layer**

Same as in "Attention, Learn to Solve Routing Problems!" (Kool et al., 2019).

In [18]:
from core.models.nds.encoder import AttentionLayer

att_layer = AttentionLayer(
    d_h=D_H,
    num_heads=N_HEADS,
    d_ff=D_FF,
)
refined_embeddings = att_layer(initial_node_embeddings)

print(f"Refined node embeddings shape: {refined_embeddings.shape}")

Refined node embeddings shape: torch.Size([4, 21, 128])


**Message Passing Layer**

In [19]:
from core.models.nds.encoder import MessagePassingLayer

MessagePassingLayer(
    d_h=D_H,
    d_ff=D_FF,
)

MessagePassingLayer(
  (W1): Linear(in_features=128, out_features=128, bias=False)
  (W2): Linear(in_features=128, out_features=128, bias=False)
  (W3): Linear(in_features=256, out_features=128, bias=False)
  (FF): Sequential(
    (0): Linear(in_features=128, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=128, bias=True)
  )
  (norm): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
)

**Tour Encoding Layer**

In [20]:
from core.models.nds.encoder import TourEncodingLayer

TourEncodingLayer(
    d_h=D_H,
    d_ff=D_FF,
)

TourEncodingLayer(
  (tour_combiner): Linear(in_features=256, out_features=128, bias=False)
  (FF): Sequential(
    (0): Linear(in_features=128, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=128, bias=True)
  )
  (norm): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
)

**All sub-layers combined into the encoder**

In [21]:
from core.models.nds.encoder import NDSEncoder
from core.models.nds.preprocessing import (
    build_encoder_input_indexes_from_sequence,
)

In [22]:
encoder = NDSEncoder().to(DEVICE)

example_sampler = CVRPSampler(
    CVRPConfig(num_customers=5, vehicle_capacity_raw=9),
    seed=42,
)
batch = example_sampler(2).to(DEVICE)
depot = batch.depot_location
customers = batch.customer_features
# routes = [[[1,3,2], [4,5]], [[2,1,4], [3,5]]]
# neighbours_index, tour_index = build_encoder_input_indexes_from_routes(routes, num_customers=batch.num_customers)

batch_solution = one_customer_per_route(batch)

neighbours_index, tour_index = build_encoder_input_indexes_from_sequence(
    batch_solution.sequence, num_customers=batch.num_customers
)

h = encoder(
    depot_location=depot,
    customer_features=customers,
    neighbours_index=neighbours_index,
    tour_index=tour_index,
)

print(
    summary(
        encoder,
        input_data={
            "depot_location": depot,
            "customer_features": customers,
            "neighbours_index": neighbours_index,
            "tour_index": tour_index,
        },
        col_names=["input_size", "output_size", "num_params"],
    )
)

Layer (type:depth-idx)                   Input Shape               Output Shape              Param #
NDSEncoder                               --                        [2, 6, 128]               --
├─CVRPInitialNodeEmbedding: 1-1          [2, 1, 2]                 [2, 6, 128]               --
│    └─Linear: 2-1                       [2, 1, 2]                 [2, 1, 128]               384
│    └─Linear: 2-2                       [2, 5, 3]                 [2, 5, 128]               512
├─Sequential: 1-2                        [2, 6, 128]               [2, 6, 128]               --
│    └─AttentionLayer: 2-3               [2, 6, 128]               [2, 6, 128]               --
│    │    └─MultiheadAttention: 3-1      [2, 6, 128]               [2, 6, 128]               66,048
│    │    └─BatchNorm1d: 3-2             [12, 128]                 [12, 128]                 256
│    │    └─Sequential: 3-3              [2, 6, 128]               [2, 6, 128]               131,712
│    │    └─BatchNorm1d

### Decoder

In [23]:
from core.models.nds.decoder import DecoderState, NDSDecoder

B, rollout_size, N = 2, 10, 4
z_dim = 10
M = 4

decoder = NDSDecoder(d_h=D_H, z_dim=Z_DIM, d_poly=D_POLY, num_heads=N_HEADS).to(DEVICE)

h_nodes = torch.randn(B, N + 1, D_H).to(DEVICE)
z = sample_seed(batch_size=B, rollout_size=rollout_size, seed_dim=z_dim, device=DEVICE)

print(f"z: {z}")

state = decoder.build_initial_state(B=B, rollout_size=rollout_size, N=N)
cache = decoder.build_cache(h_nodes=h_nodes, z=z)

selected = []
for m in range(M):
    log_p, gru_hidden_state = decoder(cache=cache, state=state)

    idx = torch.distributions.Categorical(logits=log_p).sample()
    selected.append(idx)

    new_mask = state.selected_mask.clone()
    new_mask.scatter_(2, idx.unsqueeze(-1), True)

    new_h_last_node = cache.h_nodes_expanded.gather(
        dim=2, index=idx.unsqueeze(-1).unsqueeze(-1).expand(B, rollout_size, 1, D_H)
    ).squeeze(2)
    state = DecoderState(
        h_last_node=new_h_last_node, gru_hidden=gru_hidden_state, selected_mask=new_mask
    )
selected = torch.stack(selected, dim=-1)

print(f"selected.shape: {selected.shape}")
print(f"selected:\n{selected}")

z: tensor([[[0., 0., 0., 1., 0., 0., 1., 1., 0., 0.],
         [0., 1., 1., 1., 1., 1., 1., 1., 0., 0.],
         [1., 1., 1., 1., 0., 0., 1., 1., 1., 1.],
         [1., 0., 1., 0., 0., 0., 1., 1., 1., 0.],
         [0., 1., 0., 1., 1., 0., 0., 0., 0., 1.],
         [0., 1., 1., 1., 1., 1., 0., 1., 1., 0.],
         [0., 0., 1., 1., 0., 0., 0., 1., 0., 1.],
         [1., 1., 0., 1., 0., 0., 0., 1., 0., 0.],
         [0., 0., 1., 1., 0., 1., 0., 1., 0., 1.],
         [0., 1., 0., 1., 0., 1., 1., 1., 0., 0.]],

        [[0., 1., 1., 1., 0., 0., 0., 0., 1., 0.],
         [1., 0., 1., 1., 1., 0., 0., 0., 1., 0.],
         [0., 1., 1., 0., 1., 1., 0., 0., 1., 0.],
         [1., 0., 1., 0., 1., 0., 0., 0., 1., 0.],
         [1., 1., 1., 1., 1., 1., 0., 0., 0., 0.],
         [0., 1., 0., 0., 1., 0., 1., 1., 1., 1.],
         [0., 0., 1., 0., 0., 1., 0., 0., 1., 0.],
         [0., 0., 0., 0., 1., 1., 0., 0., 1., 0.],
         [0., 1., 0., 1., 0., 1., 0., 0., 0., 0.],
         [1., 0., 1., 1., 

### Model

In [24]:
model = NDSModel().to(DEVICE)

## Training